## Mcrograd learning notebook

# Output Records
1. 2026/9/26: Adds Value class
2. 2026/10/5: Adds graphviz related method
 


In [27]:
import sys; print (sys.executable)

C:\Users\Administrator\miniforge3\envs\zero2hero\python.exe


In [28]:
import math
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline


In [29]:
from graphviz import Digraph

def trace(root):
    nodes = set()
    edges = set()

    def build(v):
        nodes.add(v)
        for child in v._prev:
            edges.add((child, v))
            trace(child)
    build(root)
    return nodes, edges

def draw_dot(root):
    dot = Digraph(format='svg', graph_attr={'rankdir' : 'LR'}) # Left to right
    nodes, edges = trace(root)

    for n in nodes:
        uid = str(id(n))
        dot.node(name=uid, label = "{data %.3f}" % (n.data, ), shape='record')
        if n._op:
            dot.node(name=uid + n._op, label = n._op)
            dot.edge(uid+n._op, uid)

    for n1, n2 in edges:
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)

    return dot


In [30]:
class Value:
    def __init__(self, data, children = (), op = '', label = ''):
        self.data = data
        self.grad = 0.0
        self._prev = set(children)
        self._op = op
        self.label = label
        # backward
        self._backward = lambda : None

    def __repr__(self):
        return f"Value(label={self.label}, data={self.data})"

    def __add__(self, other):
        out = Value(self.data + other.data, (self, other), '+')
        def backward():
            self.grad += out.grad
            other.grad += out.grad

        self._backward = backward;
        return out

    def __neg__(self):
        return self * -1;

    def __sub__(self, other):
        return self + (-other)

    def __mul__(self, other):
        out = Value(self.data * other.data, (self, other), '*')
        def backward():
            self.grad += (other.data) * out.grad
            other.grad += (self.data) * out.grad
            
        self._backward = backward;
        return out

    def __rmul__(self, other):
        return self * other;

    def __pow__(self, other):
        assert isinstance(other, (int, float)), "Only supporting int / float powers for now."
        out = Value(self.data ** other, (self, ), f'**{other}')

        def backward():
            self.grad += (other * self.data ** (other - 1.0)) * out.grad
        self._backward = backward
        return out

    def __truediv__(self, other):
        return self * other ** -1

    def tanh(self):
        x = self.data
        t = (math.exp(x * 2) - 1.0) / (math.exp(x * 2) + 1)
        out = Value(t, (self, ), 'tanh')

        def backward():
            self.grad += (1 - t**2) * out.grad
        self._backward = backward
        return out

    def exp(self):
        x = self.data;
        out = Value(math.exp(x), (self, ), 'exp')

        def backward():
            self.grad += out.data * out.grad
        self._backward = backward
        return out

    def backward(self):
        topo = []
        visited = set()
        def buildTopo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    buildTopo(child)
                topo.append(v)
        buildTopo(self);

        self.grad = 1.0
        for node in reversed(topo):
            node._backward



In [31]:
# Graph viz block, will implement later


In [32]:
a = Value(2.0, label='a')
b = Value(-3.0, label='b')
c = Value(10.0,label='c')
e = a * b; e.label = 'e'
d = e + c; d.label = 'd'
f = Value(-2.0, label='f')
L = d * f; L.label = 'L'
O = L.tanh()
O

draw_dot(O)

ExecutableNotFound: failed to execute WindowsPath('dot'), make sure the Graphviz executables are on your systems' PATH

In [ ]:
plt.plot(np.arange(-5, 5, 0.2), np.tanh(np.arange(-5, 5, 0.2))); plt.grid(); # tanh

In [ ]:
k = Value(2.0, label='k')
s = Value(4.0, label='s')
k /s
